In [1]:
import os
os.chdir("../")
%pwd

'd:\\Deep_Learning_Object_Detection\\MLOPs\\pneumonia-segmentation'

In [2]:
from dataclasses import dataclass
from pathlib import Path

@dataclass(frozen=True)
class DataIngestionConfig:
    root_dir:       Path
    ingestion_type: str            
    source:         str     
    name:           str

In [3]:
from pneumonia_segmentation.constants import *
from pneumonia_segmentation.utils.common import read_yaml, create_directories

from dotenv import load_dotenv
load_dotenv()

class ConfigurationManger:
    def __init__(self, 
                 config_filepath = CONFIG_FILE_PATH,
                 params_filepath = PARAMS_FILE_PATH):
        self.config = read_yaml(config_filepath)
        self.params = read_yaml(params_filepath)

        create_directories([self.config.artifacts_root])
    
    def _parse_data_sources(self):
        raw = os.getenv("DATA_SOURCES", "")
        
        sources = []
        for entry in raw.split(","):
            name, source_type, ingestion_type, source = entry.strip().split(":")
            sources.append({
                "name": name,
                "source_type": source_type,
                "ingestion_type": ingestion_type,
                "source": source
            })
        return sources

    def get_data_ingestion_config(self) -> DataIngestionConfig:
        config = self.config.data_ingestion_config
        create_directories([config.root_dir])
        
        sources = self._parse_data_sources()
        configs = []
        for s in sources:
            configs.append(DataIngestionConfig(
                root_dir        = config.root_dir,
                ingestion_type  = s["ingestion_type"],
                source          = s["source"],
                name            = s["name"],
            ))

        return configs

In [ ]:
import os, sys
from pneumonia_segmentation import logging
from pneumonia_segmentation.exception import CustomException
from pneumonia_segmentation.adapters.factory import IngestionAdapterFactory

class DataIngestion:
    def __init__(self, config: DataIngestionConfig):
        self.config  = config
        self.adapter = IngestionAdapterFactory.create_adapter(
            self.config.ingestion_type, self.config.source
        )
        
    def fetch_data(self) -> None:
        try: 
            dst = os.path.join(self.config.root_dir, self.config.name)
            self.adapter.fetch(dst)
            logging.info(f"Data fetched at {dst}")
        except Exception as e:
            raise CustomException(e, sys)

In [5]:
try:
    config = ConfigurationManger()
    data_ingestion_configs = config.get_data_ingestion_config()
    for config in data_ingestion_configs:
        data_ingestion = DataIngestion(config=config)
        data_ingestion.fetch_data()
except Exception as e:
    raise CustomException(e, sys)

[2026-04-17 14:18:05,741: INFO: common: yaml file: config\config.yaml loaded successfully]
[2026-04-17 14:18:05,745: INFO: common: yaml file: params.yaml loaded successfully]
[2026-04-17 14:18:05,747: INFO: common: created directory at: artifacts]
[2026-04-17 14:18:05,747: INFO: common: created directory at: artifacts/data_ingestion]
[2026-04-17 14:18:05,749: INFO: kaggle_ingestion_adapter: Downloading Kaggle dataset andrewmvd/covid19-ct-scans -> artifacts/data_ingestion\COVID-19_CT_SCANS]
[2026-04-17 14:19:34,456: INFO: 777641282: Data fetched at artifacts/data_ingestion\COVID-19_CT_SCANS]
